# Loading markers

Markers enter a simulation twice: once at the start, drawn from the initial distribution, and every step from the sources. `mpiparticles.sampling` draws them (Maxwellian velocities, the flux-weighted speeds of a plane source, quiet starts, importance sampling) and `mpiparticles.injection` has the timestep sources, which return a step's positions, velocities and weights as arrays or add them to a `ParticleStore`. Every draw takes a random generator, NumPy's or CuPy's, so a seeded run is reproducible.

In [ ]:
import numpy as np
import plasma_plots
from scipy import constants
from simgeometry.boundingbox import BoundingBox

from mpiparticles import ImportanceSampling, MarkerSpecies, ParticleStore, quiet_start
from mpiparticles.injection import (
    PlanarFluxInjector1D,
    ProfiledVolumeSource,
    cosine_profile,
)
from mpiparticles.sampling import flux_speeds, make_generator, maxwellian_velocities

plasma_plots.set_backend("plotly")
plasma_plots.set_math("latex")

rng = make_generator(1)

## Maxwellian and flux speeds

A Maxwellian at the thermal speed $v_\mathrm{th} = \sqrt{T/m}$ has normal components. The markers that cross a plane in a time $\Delta t$ are not a Maxwellian: fast ones cross more often, so the normal speeds of the flux have the density $v/v_\mathrm{th}^2\, e^{-v^2/2v_\mathrm{th}^2}$, a Rayleigh distribution. A plane source that drew half a Maxwellian would emit too slow a flux.

In [ ]:
count = 200_000
maxwellian = np.abs(maxwellian_velocities(rng, count, 1, 1.0)[0])
flux = flux_speeds(rng, count, 1.0)
edges = np.linspace(0.0, 4.5, 46)
centres = 0.5 * (edges[1:] + edges[:-1])

with plasma_plots.figure() as fig:
    ax = fig[0]
    ax.step(
        centres,
        np.histogram(maxwellian, edges, density=True)[0],
        where="mid",
        label="half Maxwellian",
    )
    ax.step(
        centres, np.histogram(flux, edges, density=True)[0], where="mid", label="flux"
    )
    ax.plot(
        centres, centres * np.exp(-0.5 * centres**2), "k--", label=r"$v\, e^{-v^2/2}$"
    )
    ax.set_xlabel(r"$|v| / v_\mathrm{th}$")
    ax.set_ylabel("probability density")
    ax.legend()

## Quiet start

Random positions and velocities carry noise of order $1/\sqrt{N}$. `quiet_start` returns a Hammersley set instead: the first coordinate evenly spaced, the others radical inverses in distinct primes, and the velocities by the inverse normal CDF. With $2^{12}$ markers on a $16 \times 16$ grid every cell holds exactly 16 markers.

In [ ]:
positions, velocities = quiet_start(1 << 12, (0.0, 0.0), (1.0, 1.0), nvel=2)
cells = np.floor(positions * 16).astype(int)
counts = np.bincount(cells[0] * 16 + cells[1], minlength=256)
print("markers per cell:", counts.min(), "to", counts.max())
print("velocity means:", velocities.mean(axis=1).round(4))

with plasma_plots.figure() as fig:
    ax = fig[0]
    ax.plot(
        positions[0, :1024], positions[1, :1024], ".", markersize=3, label="quiet start"
    )
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")

## Importance sampling

`ImportanceSampling` draws from a sampling distribution $s$ instead of the physical $f$ and gives each marker the weight factor $f/s$, so every weighted moment stays unbiased. With `temperature_ratio=4` the velocities come from a Maxwellian at $4T$: the tail above $3\,v_\mathrm{th}$ gets many more, lighter markers.

In [ ]:
plain, plain_weights = ImportanceSampling().sample_maxwellian(rng, 4000, 1, 1.0)
hot, hot_weights = ImportanceSampling(temperature_ratio=4.0).sample_maxwellian(
    rng, 4000, 1, 1.0
)
edges = np.linspace(-7.0, 7.0, 57)
centres = 0.5 * (edges[1:] + edges[:-1])
width = edges[1] - edges[0]


def weighted_density(v, w):
    hist = np.histogram(v[0], edges, weights=w)[0] / (w.sum() * width)
    return np.where(hist > 0, hist, np.nan)


with plasma_plots.figure() as fig:
    ax = fig[0]
    ax.semilogy(
        centres,
        np.exp(-0.5 * centres**2) / np.sqrt(2 * np.pi),
        "k--",
        label="Maxwellian",
    )
    ax.step(centres, weighted_density(plain, plain_weights), where="mid", label="plain")
    ax.step(
        centres, weighted_density(hot, hot_weights), where="mid", label=r"$T_s = 4T$"
    )
    ax.set_xlabel(r"$v / v_\mathrm{th}$")
    ax.set_ylabel("weighted density")
    ax.legend()
print(
    "markers above 3 v_th:",
    int(np.sum(np.abs(plain) > 3)),
    "plain,",
    int(np.sum(np.abs(hot) > 3)),
    "hot",
)

## Timestep sources

A source creates the markers of one step. `MarkerSpecies` tells it the charge, mass, marker weight, thermal speed and components of the species; `emit(dt, species)` returns the arrays and `inject(store, dt, species)` adds them to a store. The source carries the fraction of a marker it could not emit, so its long-time rate is exact. Below, a plane at $x = 0$ emits electrons at $1\ \mathrm{A/m^2}$ into a 1D3V store, and a volume source creates them with the cosine profile of an ionization region.

In [ ]:
temperature_ev = 2.0
electrons = MarkerSpecies(
    charge=-constants.e,
    mass=constants.m_e,
    weight=1.0e9,
    thermal_velocity=np.sqrt(constants.e * temperature_ev / constants.m_e),
    ndim=1,
    nvel=3,
)
store = ParticleStore(ndim=1, nvel=3)
plane = PlanarFluxInjector1D(position=0.0, current_density=1.0, direction=1.0, rng=2)
dt = 1.0e-11
created = sum(plane.inject(store, dt, electrons) for _ in range(1000))
expected = 1.0 * dt * 1000 / constants.e / electrons.weight
print(f"{created} markers, {expected:.1f} expected")

box = BoundingBox(lower_bound=(0.0,), upper_bound=(1.0e-2,))
points, values = cosine_profile(2.0e-3, 8.0e-3)
volume = ProfiledVolumeSource(box, rate=1.0e15, points=points, values=values, rng=3)
positions, velocities, weights = volume.emit(1.0e-3, electrons)

with plasma_plots.figure() as fig:
    ax = fig[0]
    ax.hist(positions[0] * 1e3, bins=40, label="volume source")
    ax.set_xlabel(r"$x$ [mm]")
    ax.set_ylabel("markers per bin")